# Compact Multi-Layer Perceptron (MLP) Baseline for HIGGS Classification

**Project:** Higgs Event Lab | **Module:** MID-03

This notebook trains the compact MLP baseline across low-level, high-level and all feature groups. It uses 250,000 rows by default to fit common notebook runtimes. Set `HIGGS_ANALYSIS_ROWS=11000000` before setup for a full-row run if your RAM/GPU budget supports it.

> **Scientific boundary:** all events are simulated; classifier metrics are not physical discovery evidence.

## Step 0 — Prepare repository, dependencies, and the original UCI data

Run the setup cell first. The helper cells retry interrupted downloads and validate the compressed file before training. The MLP training cell below uses `analysis_row_count`, which defaults to 250,000 rows.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
from packaging.requirements import Requirement
from packaging.version import Version

REPO_URL = "https://github.com/CosmicMagnetar/higgs-event-lab.git"
cwd = Path.cwd().resolve()


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
candidates = []
if os.environ.get("HIGGS_REPO_ROOT"):
    candidates.append(Path(os.environ["HIGGS_REPO_ROOT"]).expanduser().resolve())
candidates.extend([cwd, cwd / "higgs-event-lab", *cwd.parents])

def is_higgs_repo(path):
    return (path / "src" / "higgs_event_lab").is_dir() and (path / "pyproject.toml").is_file()


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
root = next((path for path in dict.fromkeys(candidates) if is_higgs_repo(path)), None)
if root is None:
    repo_target = cwd / "higgs-event-lab"
    suffix = 1
    while repo_target.exists() and not is_higgs_repo(repo_target):
        if repo_target.is_dir() and not any(repo_target.iterdir()):
            break
        suffix += 1
        repo_target = cwd / f"higgs-event-lab-{suffix}"
    print("Cloning Higgs Event Lab into:", repo_target)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_target)], check=True)
    root = repo_target.resolve()

if not is_higgs_repo(root):
    raise FileNotFoundError(f"Could not locate a valid Higgs Event Lab checkout at {root}.")


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
src_dir = root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
os.environ["HIGGS_REPO_ROOT"] = str(root)
os.chdir(root)
print("Project root:", root)


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
REQUIRED_PACKAGES = [
    ("requests", "requests>=2.31"),
    ("numpy", "numpy>=1.26"),
    ("pandas", "pandas>=2.1"),
    ("sklearn", "scikit-learn>=1.4"),
    ("matplotlib", "matplotlib>=3.8"),
]
missing_or_old = []
for module_name, requirement_text in REQUIRED_PACKAGES:
    requirement = Requirement(requirement_text)
    try:
        installed = Version(version(requirement.name))
    except PackageNotFoundError:
        installed = None
    if installed is None or (requirement.specifier and installed not in requirement.specifier):
        missing_or_old.append(requirement_text)
if missing_or_old:
    print("Installing required packages:", ", ".join(missing_or_old))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_or_old], check=True)
else:
    print("Required packages already satisfy the minimum versions.")


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
raw_path = root / "data" / "raw" / "HIGGS.csv.gz"
zip_path = raw_path.parent / "higgs_download.zip"
temp_raw_path = raw_path.with_name(raw_path.name + ".part")
validation_marker = raw_path.with_name(raw_path.name + ".validation.json")
raw_path.parent.mkdir(parents=True, exist_ok=True)
output_dir = root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
url = "https://archive.ics.uci.edu/static/public/280/higgs.zip"
full_row_count = 11_000_000
analysis_row_count = int(os.environ.get("HIGGS_ANALYSIS_ROWS", "250000"))
if not 10 <= analysis_row_count <= full_row_count:
    raise ValueError(f"HIGGS_ANALYSIS_ROWS must be between 10 and {full_row_count:,}.")
processed_path = raw_path
print("Raw dataset path:", raw_path)
print(f"Analysis rows: {analysis_row_count:,}")


## Step 0B — Resumable download helpers

This cell only defines the ZIP/GZIP validation and retry/resume functions. It does not start a download.

This section records the purpose, expected output and interpretation of the computation so the result can be checked independently.


In [ ]:
import re
import time
import zipfile
from pathlib import Path


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def find_csv_gz_member(archive_path):
    """Return the CSV.GZ member only when the ZIP directory is readable."""
    archive_path = Path(archive_path)
    if not archive_path.is_file():
        return None
    try:
        with zipfile.ZipFile(archive_path, "r") as archive:
            return next((name for name in archive.namelist() if name.lower().endswith(".csv.gz")), None)
    except (OSError, zipfile.BadZipFile, EOFError):
        return None


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _download_mode(response, existing, destination):
    if response.status_code == 416:
        Path(destination).unlink(missing_ok=True)
        raise IOError("The server rejected the resume offset; restarting from zero.")
    response.raise_for_status()
    append = existing > 0 and response.status_code == 206
    if append:
        content_range = response.headers.get("Content-Range", "")
        match = re.match(r"bytes\s+(\d+)-", content_range)
        if not match or int(match.group(1)) != existing:
            Path(destination).unlink(missing_ok=True)
            raise IOError("Server returned an unexpected byte range; restarting from zero.")
    return "ab" if append else "wb"


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _download_attempt(download_url, destination, session):
    destination = Path(destination)
    existing = destination.stat().st_size if destination.exists() else 0
    headers = {"Range": f"bytes={existing}-"} if existing else {}
    with session.get(download_url, headers=headers, stream=True, timeout=(30, 180)) as response:
        mode = _download_mode(response, existing, destination)
        expected = response.headers.get("Content-Length")
        received = 0
        with destination.open(mode) as target:
            for chunk in response.iter_content(chunk_size=4 * 1024 * 1024):
                if chunk:
                    target.write(chunk)
                    received += len(chunk)
        if expected is not None and received != int(expected):
            raise IOError(f"Incomplete HTTP response: received {received:,} of {int(expected):,} bytes.")
    if not find_csv_gz_member(destination):
        raise IOError("The downloaded ZIP is not complete yet.")


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def download_zip_resumable(download_url, destination, max_attempts=10):
    """Download with retries and HTTP Range resume where the server supports it."""
    import requests
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if find_csv_gz_member(destination):
        print("A readable ZIP is already present; reusing it.")
        return destination
    session = requests.Session()
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            _download_attempt(download_url, destination, session)
            print(f"ZIP received: {destination.stat().st_size / (1024**3):.2f} GiB.")
            return destination
        except Exception as exc:
            last_error = exc
            print(f"Download attempt {attempt}/{max_attempts} failed: {exc}")
            if attempt < max_attempts:
                time.sleep(min(2 ** (attempt - 1), 30))
    raise RuntimeError(f"Could not obtain a readable UCI ZIP. Rerun this cell. Last error: {last_error}")


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def count_gzip_rows(path, chunk_size=4 * 1024 * 1024):
    """Fully read a gzip file and count CSV rows without loading it into memory."""
    import gzip
    newline_count = 0
    last_byte = b""
    with gzip.open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            newline_count += chunk.count(b"\n")
            last_byte = chunk[-1:]
    if last_byte and last_byte != b"\n":
        newline_count += 1
    return newline_count


## Step 0C — Download, extract, and validate the raw HIGGS file

This cell reuses a previously validated file, repairs old partial files, or downloads and validates the complete 11-million-row source. Rerun this cell after a network failure.

In [ ]:
import json
import shutil
import zipfile


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _write_validation_marker(path, rows):
    marker = Path(path).with_name(Path(path).name + ".validation.json")
    marker.write_text(
        json.dumps({"rows": int(rows), "size_bytes": int(Path(path).stat().st_size)}, indent=2),
        encoding="utf-8",
    )


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _existing_raw_is_valid(path, expected_rows, marker_path):
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        return False
    try:
        metadata = json.loads(Path(marker_path).read_text(encoding="utf-8"))
        if metadata.get("rows") == expected_rows and metadata.get("size_bytes") == path.stat().st_size:
            with path.open("rb") as handle:
                return handle.read(2) == b"\x1f\x8b"
    except (OSError, ValueError, TypeError):
        pass
    try:
        rows = count_gzip_rows(path)
        if rows != expected_rows:
            print(f"Existing raw file is incomplete: expected {expected_rows:,}, found {rows:,} rows.")
            return False
        _write_validation_marker(path, rows)
        return True
    except Exception as exc:
        print("Existing raw file failed gzip validation:", exc)
        return False


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _extract_zip_member(zip_path, member_name, temp_path):
    temp_path.unlink(missing_ok=True)
    with zipfile.ZipFile(zip_path, "r") as archive:
        with archive.open(member_name, "r") as source, temp_path.open("wb") as target:
            shutil.copyfileobj(source, target, length=4 * 1024 * 1024)
    return count_gzip_rows(temp_path)


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _extract_and_validate_raw(url, zip_path, raw_path, temp_raw_path, marker_path, expected_rows):
    gz_name = find_csv_gz_member(zip_path)
    if not gz_name:
        print("No complete ZIP found; downloading or resuming the UCI archive.")
        download_zip_resumable(url, zip_path, max_attempts=10)
        gz_name = find_csv_gz_member(zip_path)
    if not gz_name:
        raise RuntimeError("No readable .csv.gz member found. Rerun the download helper cell.")
    print("Extracting dataset member:", gz_name)
    try:
        rows = _extract_zip_member(zip_path, gz_name, temp_raw_path)
        if rows != expected_rows:
            raise ValueError(f"Expected {expected_rows:,} rows, found {rows:,}.")
        temp_raw_path.replace(raw_path)
        _write_validation_marker(raw_path, rows)
    except Exception as exc:
        temp_raw_path.unlink(missing_ok=True)
        zip_path.unlink(missing_ok=True)
        raise RuntimeError("Extraction/validation failed. Incomplete files were removed; rerun this cell.") from exc
    zip_path.unlink(missing_ok=True)
    return rows


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
complete_row_count = None
if raw_path.exists() and _existing_raw_is_valid(raw_path, full_row_count, validation_marker):
    complete_row_count = full_row_count
    print("A previously validated complete HIGGS file is present; skipping download.")
else:
    raw_path.unlink(missing_ok=True)
    validation_marker.unlink(missing_ok=True)
    complete_row_count = _extract_and_validate_raw(
        url, zip_path, raw_path, temp_raw_path, validation_marker, full_row_count
    )
if complete_row_count != full_row_count:
    raise ValueError(f"Dataset has {complete_row_count:,} rows; expected {full_row_count:,}.")
print("Raw HIGGS data ready:", raw_path)
print(f"Validated row count: {complete_row_count:,}")
print(f"Compressed size: {raw_path.stat().st_size / (1024**3):.2f} GiB")


## 1. Environment and Package Initialization

**Question:** Which libraries and modules are required to run the deep learning benchmark?

**Why it matters:** To guarantee strict experimental reproducibility and modular design, all model layers, training loops, and data loading contracts reside in tested package modules rather than loose notebook script blocks.

**Explanation:** We import PyTorch for building the neural network, alongside NumPy and Pandas for tensor operations. We also import our project modules `features`, `protocol`, and `neural` to enforce schema constraints and split isolation.

**Expected output:** Confirmation of the PyTorch version and available device acceleration.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from higgs_event_lab.features import FEATURE_GROUPS, get_feature_columns
from higgs_event_lab.protocol import SplitConfig, create_stratified_splits
from higgs_event_lab.neural import MLPConfig, build_mlp, run_feature_group_benchmarks

print("PyTorch version:", torch.__version__)
print("CUDA accelerator available:", torch.cuda.is_available())


## 2. Multi-Layer Perceptron Architecture and Regularization

**Question:** How is the neural network structured to prevent overfitting and ensure fast, stable convergence?

**Why it matters:** Unregularized deep networks easily overfit high-dimensional collision data. Batch normalization stabilizes internal covariate shift during gradient updates, while dropout prevents co-adaptation of hidden features.

**Explanation:** The compact MLP architecture applies sequential hidden blocks containing a linear projection, one-dimensional batch normalization, ReLU activation, and twenty percent dropout. The final layer outputs an unconstrained binary logit.

**Expected output:** Layer-by-layer architectural printout and parameter counts.

In [ ]:
config = MLPConfig(hidden_sizes=(64, 32), dropout=0.20, learning_rate=1e-3)
sample_model = build_mlp(input_features=28, config=config)
total_params = sum(param.numel() for param in sample_model.parameters())
print(sample_model)
print("Total network parameters:", total_params)


## 3. The 21 Low-Level and 7 High-Level Feature Groups

**Question:** How are the twenty-eight input variables partitioned across kinematic and derived feature groups?

**Why it matters:** The central scientific question of the benchmark is evaluating whether neural networks require human-engineered invariant masses or can learn equivalent or superior decision boundaries directly from raw detector quantities.

**Explanation:** Low-level features comprise momenta, pseudorapidity, azimuthal angles, and b-tagging metrics. High-level features comprise invariant masses calculated from reconstructed decay subsystems.

**Expected output:** Verified counts and column names for the low-level, high-level, and complete feature sets.

In [ ]:
for group_name in ("low_level", "high_level", "all"):
    columns = get_feature_columns(group_name)
    print(f"{group_name:<11}: {len(columns)} features -> {list(columns[:3])}...")


## 4. Reproducible stratified splitting without data leakage

**Question:** How do we partition all approximately 11 million events into training, validation and held-out test subsets fairly?

**Why it matters:** The complete-data benchmark uses the shared 70% / 15% / 15% stratified protocol with seed 42. Scaling the row count changes the measured result, so the source and exact count are recorded in every artifact.

**Expected output:** Exact full-data row counts for each disjoint partition.


In [ ]:
from higgs_event_lab.data import load_higgs_sample
import pandas as pd

labels_frame = pd.read_csv(
    processed_path,
    header=None,
    usecols=[0],
    names=["class_label"],
    nrows=analysis_row_count,
    dtype={"class_label": "float32"},
)
labels_float = labels_frame["class_label"].to_numpy(dtype=np.float32)
if not np.isin(labels_float, [0.0, 1.0]).all():
    raise ValueError("HIGGS labels must be 0.0 or 1.0; inspect the raw file before training.")
labels = labels_float.astype(np.int64)
split = create_stratified_splits(labels, SplitConfig(seed=42))
print(f"Dataset rows selected for analysis: {len(labels):,}")
print("Features available:", len(FEATURE_GROUPS["all"]))
print(f"Signal fraction: {labels.mean():.4f}")
print(f"Split sizes — Train: {len(split.train):,}  Val: {len(split.validation):,}  Test: {len(split.test):,}")
del labels_frame, labels, split


## 5. Training Across All Three Feature Sets

**Question:** How does model performance vary across low-level, high-level, and combined feature representations?

**Why it matters:** Comparing model metrics across the three feature groups reveals the relative importance of engineered domain features versus raw sensor measurements.

**Explanation:** We train identical MLP architectures across each subset using AdamW with early stopping monitored on validation ROC-AUC. When training halts, the best historical model weights are automatically restored.

**Expected output:** Completed training epoch count and test evaluation metrics for each feature group.

In [ ]:
config = MLPConfig(
    hidden_sizes=(64, 32),
    dropout=0.20,
    learning_rate=1e-3,
    weight_decay=1e-4,
    max_epochs=30,
    early_stopping_patience=5,
    batch_size=1024,
    seed=42,
    device="auto",
)
benchmark_results = run_feature_group_benchmarks(
    data_path=processed_path,
    row_count=analysis_row_count,
    output_base_dir=output_dir,
    config=config,
)
for group_name, result in benchmark_results.items():
    metrics = result["test"]["metrics"]
    print(f"{group_name:<11} epochs={len(result['history']):2d} device={result['device']:<4} "
          f"ROC-AUC={metrics['roc_auc']:.4f} PR-AUC={metrics['pr_auc']:.4f}")


## 6 — Artifact Export for Dashboard and Reports

The benchmark helper writes each measured MLP run to `outputs/mlp_*` with a `metrics.json` file containing genuine training history and a `test_predictions.csv` file containing held-out predictions.


## 6.1 Export learning curves

The training module stores genuine per-epoch loss and validation ROC-AUC in each MLP `metrics.json`. This cell plots those saved histories for direct inclusion in the report.


In [ ]:
import json
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for group_name in ("low_level", "high_level", "all"):
    metadata = json.loads((output_dir / f"mlp_{group_name}" / "metrics.json").read_text())
    history = metadata["training"]["history"]
    epochs = [item["epoch"] for item in history]
    axes[0].plot(epochs, [item["train_loss"] for item in history], label=group_name)
    axes[1].plot(epochs, [item["validation_roc_auc"] for item in history], label=group_name)
axes[0].set(xlabel="Epoch", ylabel="Training loss", title="MLP training loss")
axes[1].set(xlabel="Epoch", ylabel="Validation ROC-AUC", title="MLP validation learning")
for axis in axes:
    axis.legend()
fig.tight_layout()
fig.savefig(output_dir / "mlp_learning_curves.png", dpi=160)
plt.show()
print("Learning curves saved to:", output_dir / "mlp_learning_curves.png")


## Step 7 — Verify exported analysis-sample artifacts

This section records the purpose, expected output and interpretation of the computation so the result can be checked independently.


In [ ]:
expected_run_ids = ["mlp_low_level", "mlp_high_level", "mlp_all"]

print("Artifact verification:")
all_ok = True
for run_id in expected_run_ids:
    run_dir = output_dir / run_id
    metadata_ok = (run_dir / "metrics.json").is_file()
    predictions_ok = (run_dir / "test_predictions.csv").is_file()
    all_ok = all_ok and metadata_ok and predictions_ok
    print(f"  {run_id:<18} metrics.json={metadata_ok} predictions={predictions_ok}")

if not all_ok:
    raise FileNotFoundError("One or more MLP artifacts are missing; rerun the training cell.")
print("All three measured MLP analysis-sample artifacts are ready for the report and dashboard.")
